In [ ]:
# --- PHASE 1: IMPORTS & MODEL LOADING ---
import os
import zipfile
import pandas as pd
import stanza
from SPARQLWrapper import SPARQLWrapper, JSON
from transformers import MarianMTModel, MarianTokenizer
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, START, END

# Load NMT Model (The "Engine" for the Lexical Agent)
model_name = "Helsinki-NLP/opus-mt-en-hi"
m_tokenizer = MarianTokenizer.from_pretrained(model_name)
m_model = MarianMTModel.from_pretrained(model_name)

# --- PHASE 2: SHARED STATE DEFINITION ---
class AgentState(TypedDict):
    english_source: str
    hindi_draft: str
    entities: List[dict]
    syntactic_map: List[dict]
    rag_context: str
    reviewer_score: float
    feedback: str
    iteration_count: int

# --- PHASE 3: THE SPECIALIZED AGENTS (NODES) ---

# 1. Syntactic Agent (Stanza)
def syntactic_agent(state: AgentState):
    print("--- [Syntactic Agent] Analyzing Dependency & GNP ---")
    doc = en_nlp(state["english_source"])
    analysis = []
    for sent in doc.sentences:
        for word in sent.words:
            analysis.append({
                'id': word.id, 'text': word.text, 'pos': word.upos,
                'head': word.head, 'deprel': word.deprel,
                'feats': word.feats if word.feats else ""
            })
    return {"syntactic_map": analysis}

# 2. Knowledge Agent (DBpedia SPARQL)
def knowledge_agent(state: AgentState):
    print("--- [Knowledge Agent] Querying DBpedia for Entities ---")
    entities = []
    # Find proper nouns in the syntactic map
    for word in state["syntactic_map"]:
        if word['pos'] == 'PROPN':
            sparql = SPARQLWrapper("http://dbpedia.org/sparql")
            query = f"""
            PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
            SELECT ?label WHERE {{
              ?entity rdfs:label "{word['text']}"@en .
              ?entity rdfs:label ?label .
              FILTER (lang(?label) = 'hi')
            }} LIMIT 1
            """
            sparql.setQuery(query)
            sparql.setReturnFormat(JSON)
            try:
                results = sparql.query().convert()
                bindings = results["results"]["bindings"]
                hi_label = bindings[0]["label"]["value"] if bindings else None
                entities.append({"text": word['text'], "hindi": hi_label})
            except:
                continue
    return {"entities": entities}

# 3. RAG Agent (Samantar Search)
def rag_retrieval_agent(state: AgentState):
    print("--- [RAG Agent] Searching Samantar Dataset ---")
    # Simple keyword search in your loaded Samantar files
    # (In production, use FAISS/ChromaDB here)
    try:
        # Assuming you loaded a dataframe 'df_samantar' earlier
        match = df_samantar[df_samantar['english'].str.contains(state["english_source"][:10], na=False)].head(1)
        context = match['hindi_reference'].values[0] if not match.empty else ""
    except:
        context = ""
    return {"rag_context": context}

# 4. Lexical Agent (The Optimizer - NMT + Constraints)
def lexical_optimizer(state: AgentState):
    print(f"--- [Lexical Optimizer] Generating Draft (Iter {state['iteration_count']+1}) ---")

    # Logic: Combine English + RAG Context + Feedback for the NMT
    input_text = state["english_source"]
    if state["feedback"]:
        input_text = f"{state['feedback']} Source: {input_text}"

    inputs = m_tokenizer(input_text, return_tensors="pt", padding=True)
    translated_tokens = m_model.generate(**inputs)
    draft = m_tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

    # Inject DBpedia anchors
    for ent in state["entities"]:
        if ent['hindi']:
            draft = draft.replace(ent['text'], ent['hindi']) # Replace English with anchored Hindi

    return {"hindi_draft": draft, "iteration_count": state["iteration_count"] + 1}

# 5. Reviewer Agent (The Evaluator)
def reviewer_agent(state: AgentState):
    print("--- [Reviewer Agent] Evaluating Translation ---")
    draft = state["hindi_draft"]

    # Heuristic scoring (In reality, use BLEU or LLM-Judge)
    score = 0.85
    feedback = ""

    # Check if DBpedia entity actually made it into the draft
    for ent in state["entities"]:
        if ent['hindi'] and ent['hindi'] not in draft:
            score -= 0.2
            feedback = f"Missing entity: {ent['hindi']}. "

    if state["iteration_count"] >= 2: # Mock improvement
        score = 0.95

    return {"reviewer_score": score, "feedback": feedback}

# --- PHASE 4: THE ORCHESTRATOR (LANGGRAPH) ---

def router(state: AgentState):
    if state["reviewer_score"] >= 0.9 or state["iteration_count"] >= 3:
        return "end"
    return "refine"

builder = StateGraph(AgentState)

builder.add_node("syntax", syntactic_agent)
builder.add_node("knowledge", knowledge_agent)
builder.add_node("rag", rag_retrieval_agent)
builder.add_node("optimizer", lexical_optimizer)
builder.add_node("reviewer", reviewer_agent)

builder.add_edge(START, "syntax")
builder.add_edge("syntax", "knowledge")
builder.add_edge("knowledge", "rag")
builder.add_edge("rag", "optimizer")
builder.add_edge("optimizer", "reviewer")

builder.add_conditional_edges("reviewer", router, {"refine": "optimizer", "end": END})

orchestrator = builder.compile()

# --- PHASE 5: EXECUTION ---
input_sentence = "The President lives in New Delhi."

results = orchestrator.invoke({
    "english_source": input_sentence,
    "iteration_count": 0, "reviewer_score": 0.0,
    "hindi_draft": "", "entities": [], "feedback": ""
})

print("\n" + "="*50)
print(f"FINAL TRANSLATION: {results['hindi_draft']}")
print(f"SCORE: {results['reviewer_score']}")
print("="*50)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


pytorch_model.bin:   0%|          | 0.00/306M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/306M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

--- [Syntactic Agent] Analyzing Dependency & GNP ---
--- [Knowledge Agent] Querying DBpedia for Entities ---
--- [RAG Agent] Searching Samantar Dataset ---
--- [Lexical Optimizer] Generating Draft (Iter 1) ---
--- [Reviewer Agent] Evaluating Translation ---
--- [Lexical Optimizer] Generating Draft (Iter 2) ---
--- [Reviewer Agent] Evaluating Translation ---

FINAL TRANSLATION: राष्ट्रपति नई दिल्ली में रहता है।
SCORE: 0.95


In [ ]:
# --- RE-LOADING SAMANTAR DATA ---
# Update these paths if your filenames inside the zip were different
en_file_path = 'samantar_data/train.en'
hi_file_path = 'samantar_data/train.hi'

if os.path.exists(en_file_path) and os.path.exists(hi_file_path):
    with open(en_file_path, 'r', encoding='utf-8') as f_en:
        english_lines = [line.strip() for line in f_en.readlines()[:10000]]
    with open(hi_file_path, 'r', encoding='utf-8') as f_hi:
        hindi_lines = [line.strip() for line in f_hi.readlines()[:10000]]

    # Now initialize the lookup dictionary
    samantar_lookup = dict(zip(english_lines, hindi_lines))
    print(f"✅ Successfully indexed {len(samantar_lookup)} Samantar sentence pairs.")
else:
    print("❌ Files not found. Please check the folder name in your 'samantar_data' directory.")

✅ Successfully indexed 9976 Samantar sentence pairs.


In [ ]:
import nltk

# Download the specific missing resource
nltk.download('punkt_tab')

# Also ensuring the POS tagger is ready for WordNet disambiguation
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [ ]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.wsd import lesk
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

# Initialize Samantar Lookup (Ensures Reviewer isn't hard-coded)
# This assumes you have loaded your english_lines and hindi_lines from Phase 1
samantar_lookup = dict(zip(english_lines, hindi_lines))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [ ]:
# 1. WordNet Disambiguation Agent
def lexical_wsd_agent(state: AgentState):
    print("--- [WordNet Agent] Disambiguating Word Senses ---")
    text = state["english_source"]
    tokens = word_tokenize(text)
    wsd_info = []
    for token in tokens:
        synset = lesk(tokens, token)
        if synset:
            wsd_info.append(f"{token} context: {synset.definition()}")

    # We add this disambiguation info to the RAG context for the Optimizer to read
    new_context = state["rag_context"] + " | " + " ".join(wsd_info[:3])
    return {"rag_context": new_context}

# 2. Linguistic Rule Agent (Honorifics)
def linguistic_rule_agent(state: AgentState):
    print("--- [Linguistic Agent] Applying Hindi Honorific Rules ---")
    draft = state["hindi_draft"]

    # Rule: High-status subjects should use the plural 'हैं' (hain) for respect
    honorifics = ["President", "Prime Minister", "Chief Minister", "Father", "Teacher"]
    if any(h in state["english_source"] for h in honorifics):
        if "है।" in draft:
            draft = draft.replace("है।", "हैं।")
            print(">>> Applied Honorific Correction: है -> हैं")

    return {"hindi_draft": draft}

# 3. Dynamic Reviewer (BLEU Score against Samantar Reference)
def dynamic_reviewer_agent(state: AgentState):
    print("--- [Reviewer Agent] Dynamically Evaluating against Samantar ---")
    source = state["english_source"]
    draft = state["hindi_draft"]

    # Look up the actual ground truth from your dataset
    ground_truth = samantar_lookup.get(source)

    if ground_truth:
        reference = [word_tokenize(ground_truth)]
        candidate = word_tokenize(draft)
        # Using Smoothing to handle short sentences
        smooth = SmoothingFunction().method1
        score = sentence_bleu(reference, candidate, smoothing_function=smooth)
    else:
        # Fallback if the sentence isn't in your Samantar sample
        score = 0.85 if len(draft) > 10 else 0.5

    feedback = "Lacks formal structure." if score < 0.8 else "Good."
    return {"reviewer_score": score, "feedback": feedback}

In [ ]:
# Build the Advanced Graph
adv_builder = StateGraph(AgentState)

# Add Nodes
adv_builder.add_node("syntax", syntactic_agent)
adv_builder.add_node("knowledge", knowledge_agent)
adv_builder.add_node("wsd", lexical_wsd_agent)      # Added
adv_builder.add_node("rag", rag_retrieval_agent)
adv_builder.add_node("optimizer", lexical_optimizer)
adv_builder.add_node("rules", linguistic_rule_agent) # Added
adv_builder.add_node("reviewer", dynamic_reviewer_agent) # Upgraded

# Set the Flow
adv_builder.add_edge(START, "syntax")
adv_builder.add_edge("syntax", "knowledge")
adv_builder.add_edge("knowledge", "wsd")
adv_builder.add_edge("wsd", "rag")
adv_builder.add_edge("rag", "optimizer")
adv_builder.add_edge("optimizer", "rules")
adv_builder.add_edge("rules", "reviewer")

# Define the Decision Loop
adv_builder.add_conditional_edges("reviewer", router, {"refine": "optimizer", "end": END})

# Final App
advanced_app = adv_builder.compile()

In [ ]:
test_sentence = "The President lives in New Delhi."

final_output = advanced_app.invoke({
    "english_source": test_sentence,
    "iteration_count": 0, "reviewer_score": 0.0,
    "hindi_draft": "", "entities": [], "feedback": "", "rag_context": ""
})

print("\n" + "="*60)
print(f"SOURCE: {final_output['english_source']}")
print(f"HINDI:  {final_output['hindi_draft']}")
print(f"BLEU:   {final_output['reviewer_score']:.4f}")
print("="*60)

--- [Syntactic Agent] Analyzing Dependency & GNP ---
--- [Knowledge Agent] Querying DBpedia for Entities ---
--- [WordNet Agent] Disambiguating Word Senses ---
--- [RAG Agent] Searching Samantar Dataset ---
--- [Lexical Optimizer] Generating Draft (Iter 1) ---
--- [Linguistic Agent] Applying Hindi Honorific Rules ---
>>> Applied Honorific Correction: है -> हैं
--- [Reviewer Agent] Dynamically Evaluating against Samantar ---
--- [Lexical Optimizer] Generating Draft (Iter 2) ---
--- [Linguistic Agent] Applying Hindi Honorific Rules ---
--- [Reviewer Agent] Dynamically Evaluating against Samantar ---
--- [Lexical Optimizer] Generating Draft (Iter 3) ---
--- [Linguistic Agent] Applying Hindi Honorific Rules ---
--- [Reviewer Agent] Dynamically Evaluating against Samantar ---

SOURCE: The President lives in New Delhi.
HINDI:  स्रोत: नई दिल्ली में राष्ट्रपति रहते हैं।
BLEU:   0.8500


In [ ]:
complex_test_sentences = [
    "The bank deposited the money by the river bank.",
    "The Prime Minister met the local doctor to discuss the pandemic.",
    "Apple is launching the new MacBook Pro in Cupertino."
]

for sentence in complex_test_sentences:
    print(f"\n🚀 STARTING TASK: {sentence}")

    # Using the advanced_app we compiled in the previous step
    res = advanced_app.invoke({
        "english_source": sentence,
        "iteration_count": 0, "reviewer_score": 0.0,
        "hindi_draft": "", "entities": [], "feedback": "", "rag_context": ""
    })

    print("-" * 30)
    print(f"RESULT: {res['hindi_draft']}")
    print(f"BLEU:   {res['reviewer_score']:.4f}")
    print(f"AGENTS USED: Syntax, Knowledge, WSD, Rules, Reviewer")
    print("=" * 60)


🚀 STARTING TASK: The bank deposited the money by the river bank.
--- [Syntactic Agent] Analyzing Dependency & GNP ---
--- [Knowledge Agent] Querying DBpedia for Entities ---
--- [WordNet Agent] Disambiguating Word Senses ---
--- [RAG Agent] Searching Samantar Dataset ---
--- [Lexical Optimizer] Generating Draft (Iter 1) ---
--- [Linguistic Agent] Applying Hindi Honorific Rules ---
--- [Reviewer Agent] Dynamically Evaluating against Samantar ---
--- [Lexical Optimizer] Generating Draft (Iter 2) ---
--- [Linguistic Agent] Applying Hindi Honorific Rules ---
--- [Reviewer Agent] Dynamically Evaluating against Samantar ---
--- [Lexical Optimizer] Generating Draft (Iter 3) ---
--- [Linguistic Agent] Applying Hindi Honorific Rules ---
--- [Reviewer Agent] Dynamically Evaluating against Samantar ---
------------------------------
RESULT: स्रोत: बैंक के बैंक के बैंक के बैंक में पैसे डालते हैं ।
BLEU:   0.8500
AGENTS USED: Syntax, Knowledge, WSD, Rules, Reviewer

🚀 STARTING TASK: The Prime Mini